# 01 — Ingest: Databricks Docs Corpus
## Databricks Expert Agent Project

**What this notebook does:**
1. Sets up Unity Catalog schema and volume
2. Pulls every Databricks doc page URL from the official sitemap
3. Scrapes them in parallel using Spark workers (distributed — this is the Databricks way)
4. Writes the raw text to a managed Delta table in Unity Catalog

**Output:** `chatbot.rag_chatbot.raw_docs` — the foundation the agent learns from

## 1. Configure Target Unity Catalog Assets

This cell defines the Unity Catalog location where the Databricks documentation corpus will live. The notebook writes raw scraped pages into `chatbot.rag_chatbot.raw_docs` and uses the `raw_docs` volume as the landing area for optional internal wiki content.

Keep these values aligned with the app configuration. The downstream chunking and Vector Search notebooks expect the same catalog, schema, and table names.

In [0]:
# ── Project config ──────────────────────────────────────────────
CATALOG     = "chatbot"
SCHEMA      = "rag_chatbot"
VOLUME      = "raw_docs"

# Derived — don't change these
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
RAW_TABLE   = f"{CATALOG}.{SCHEMA}.raw_docs"

print(f"Catalog : {CATALOG}")
print(f"Schema  : {CATALOG}.{SCHEMA}")
print(f"Volume  : {VOLUME_PATH}")
print(f"Table   : {RAW_TABLE}")

## Cluster Dependencies
The following libraries must be installed at the **cluster level** (not notebook level):
- `requests` — HTTP calls to fetch doc pages
- `beautifulsoup4` — HTML parsing
- `lxml` — parsing engine for BeautifulSoup

**To install:** Compute → your cluster → Libraries tab → Install New → PyPI

## 3. Create Catalog, Schema, And Volume

This cell creates the Unity Catalog assets if they do not already exist. The catalog and schema hold the managed Delta tables, while the volume is used as a file landing area for optional internal markdown content.

Running this cell is idempotent: if the assets already exist, Databricks leaves them in place.

In [0]:
spark.sql(f'create catalog if not exists {CATALOG}')
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME  IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")
print(f'✓ Catalog: {CATALOG}')
print(f"✓ Schema : {CATALOG}.{SCHEMA}")
print(f"✓ Volume : {VOLUME_PATH}")

## 4. Import Libraries

This cell imports the libraries used by the ingestion flow:

- `requests` fetches Microsoft Learn TOCs and HTML pages.
- `BeautifulSoup` extracts readable page content from HTML.
- `datetime` stamps each scraped row.
- `os` and `re` support optional internal wiki ingestion and markdown cleanup.

In [0]:
import requests
from bs4 import BeautifulSoup
from datetime import datetime
import os
import re

## 5. Discover Azure Databricks Documentation URLs

This cell reads the official Azure Databricks Microsoft Learn table of contents and turns every valid Databricks documentation link into a normalized URL.

The important work here is URL hygiene: the notebook removes fragments and query strings, resolves relative links, filters out non-Databricks URLs, and deduplicates the final list before scraping.

In [0]:
# ── Azure Databricks TOC crawl ──────────────────────────────────
TOC_URL  = "https://learn.microsoft.com/en-us/azure/databricks/toc.json"
BASE_URL = "https://learn.microsoft.com/en-us/azure/databricks/"

def normalize_href(href: str):
    if not href:
        return None

    href = href.split("?")[0].split("#")[0].strip()
    if not href:
        return None

    if href.startswith("https://") or href.startswith("http://"):
        if "learn.microsoft.com" in href and "/azure/databricks" in href:
            return href.rstrip("/")
        return None

    if href.startswith("/en-us/azure/databricks"):
        return "https://learn.microsoft.com" + href.rstrip("/")

    if href.startswith("/azure/databricks"):
        return "https://learn.microsoft.com/en-us" + href.rstrip("/")

    if not href.startswith("/"):
        return BASE_URL.rstrip("/") + "/" + href.lstrip("/").rstrip("/")

    return None

def extract_urls(node, urls=None):
    if urls is None:
        urls = []

    if isinstance(node, dict):
        href = node.get("href")
        normalized = normalize_href(href) if href else None

        if normalized and not node.get("redirect"):
            urls.append(normalized)

        for key in ("items", "children"):
            for child in node.get(key, []):
                extract_urls(child, urls)

    elif isinstance(node, list):
        for item in node:
            extract_urls(item, urls)

    return urls

print(f"Fetching TOC from {TOC_URL} ...")
resp = requests.get(
    TOC_URL,
    timeout=30,
    headers={"User-Agent": "DatabricksAgentProject/1.0"}
)
resp.raise_for_status()

toc = resp.json()
raw_urls = extract_urls(toc)

seen = set()
ms_urls = []
for u in raw_urls:
    if u not in seen:
        seen.add(u)
        ms_urls.append(u)

all_doc_urls = ms_urls

print(f"Azure Databricks MS Learn docs: {len(ms_urls):,}")
print("Ready to scrape.")
print(ms_urls[:10])

## 6. Define The HTML Page Scraper

This cell defines the function that turns one documentation URL into one structured row.

For each page, the function:

- Downloads the HTML with a project-specific user agent.
- Removes noisy tags such as scripts, images, and SVGs.
- Finds the main article content when possible.
- Collapses whitespace into clean text.
- Returns a status value so failures can be tracked instead of stopping the whole scrape.

In [0]:
# ── Page scraper function ───────────────────────────────────────
def scrape_doc_page(url: str):
    try:
        r = requests.get(
            url,
            timeout=30,
            headers={"User-Agent": "DatabricksAgentProject/1.0"}
        )
        r.raise_for_status()

        soup = BeautifulSoup(r.text, "lxml")

        for tag in soup(["script", "style", "noscript", "svg", "img"]):
            tag.decompose()

        title = soup.title.get_text(" ", strip=True) if soup.title else url

        main = (
            soup.find("main")
            or soup.find("article")
            or soup.find("div", {"role": "main"})
            or soup.body
        )

        content = main.get_text(" ", strip=True) if main else ""
        content = " ".join(content.split())

        return (
            url,
            title,
            content,
            "ok",
            datetime.utcnow().isoformat()
        )

    except Exception as e:
        return (
            url,
            None,
            None,
            f"error: {str(e)[:500]}",
            datetime.utcnow().isoformat()
        )

## 7. Prepare Distributed Scraping With Spark

This cell wraps the Python scraper as a Spark UDF and creates a DataFrame of URLs to process.

The reason for using Spark here is scale: each URL can be scraped independently, so Databricks can distribute the work across cluster workers instead of scraping every page serially on the driver.

In [0]:
from pyspark.sql.functions import udf, col
from pyspark.sql.types import StructType, StructField, StringType

# Define the return schema for our scraper UDF
result_schema = StructType([
    StructField("url",          StringType(), True),
    StructField("title",        StringType(), True),
    StructField("content",      StringType(), True),
    StructField("status",       StringType(), True),
    StructField("scraped_date", StringType(), True),
])

# Wrap scraper function as a Spark UDF
scrape_udf = udf(scrape_doc_page, result_schema)

# Build URL DataFrame
urls_df = spark.createDataFrame([(u,) for u in all_doc_urls], ["url"])

print(f"Scraping {urls_df.count():,} URLs across workers...")
print("Expect several minutes depending on cluster size.\n")

## 8. Run The Scrape And Separate Successes From Failures

This cell executes the scraper across the URL DataFrame and splits the result into two cached DataFrames:

- `ok_df` contains pages that were successfully scraped.
- `bad_df` contains page-level failures and error messages.

Keeping failures in a table-shaped result makes the pipeline auditable and lets the successful pages continue into the raw corpus.

In [0]:
from datetime import datetime

# Apply the scraper UDF across the URL DataFrame. Spark can distribute these
# independent HTTP fetches across workers, which is much faster than scraping
# every documentation page serially on the driver.
scraped_df = urls_df.select(scrape_udf(col("url")).alias("result")).select("result.*")

# Split successful rows from failures so the good pages can continue while the
# failure table remains available for troubleshooting and quality review.
ok_df  = scraped_df.filter(col("status") == "ok").cache()
bad_df = scraped_df.filter(col("status") != "ok").cache()

print(f"✓ Successful : {ok_df.count():,}")
print(f"✗ Failed     : {bad_df.count():,}")

## 9. Inspect Page-Level Failures

This cell summarizes scrape failures by error text. Use it to identify systematic issues such as redirects, retired pages, throttling, or pages whose HTML shape changed.

A small number of failures can be acceptable, but repeated errors should be reviewed before refreshing the production retrieval index.

In [0]:
bad_df.select("status").groupBy("status").count().display()

## 10. Optionally Add Internal Azure Databricks Wiki Content

This cell is optional. If a cloned internal wiki is present in the configured Unity Catalog volume, the notebook reads markdown files, strips markdown formatting noise, and converts them into rows that match the Microsoft Learn scrape schema.

The value is that internal TSGs and field notes can be retrieved beside official docs, while still remaining identifiable through the `internal_wiki://` URL pattern.

In [0]:
# ── Internal ADO Wiki / TSG Markdown ingestion ──────────────────────────────
# Upload your cloned wiki repo folder into this Volume path first.
# Example final path:
# /Volumes/chatbot/rag_chatbot/raw_docs/internal_wiki/AzureDataBricks.wiki

WIKI_ROOT = f"{VOLUME_PATH}/internal_wiki/AzureDataBricks.wiki"

def clean_markdown(text: str) -> str:
    # Keep the useful content, remove markdown noise
    text = re.sub(r"```.*?```", " ", text, flags=re.DOTALL)      # code blocks
    text = re.sub(r"`([^`]*)`", r"\1", text)                     # inline code markers
    text = re.sub(r"!\[.*?\]\(.*?\)", " ", text)                 # images
    text = re.sub(r"\[(.*?)\]\((.*?)\)", r"\1", text)            # links keep text
    text = re.sub(r"#+\s*", "", text)                            # headings
    text = re.sub(r"[*_~]", "", text)                            # markdown emphasis
    text = re.sub(r"\n{2,}", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()

wiki_rows = []

if os.path.exists(WIKI_ROOT):
    for root, dirs, files in os.walk(WIKI_ROOT):
        for file in files:
            if not file.lower().endswith((".md", ".markdown")):
                continue

            full_path = os.path.join(root, file)
            rel_path = os.path.relpath(full_path, WIKI_ROOT)

            try:
                with open(full_path, "r", encoding="utf-8", errors="ignore") as f:
                    raw_md = f.read()

                content = clean_markdown(raw_md)

                if len(content) < 100:
                    continue

                title = rel_path.replace("\\", "/").replace(".md", "")
                url = f"internal_wiki://AzureDataBricks.wiki/{title}"

                wiki_rows.append((url, title, content, "ok", datetime.utcnow().isoformat()))

            except Exception as e:
                wiki_rows.append((
                    f"internal_wiki://AzureDataBricks.wiki/{rel_path}",
                    rel_path,
                    None,
                    f"error: {str(e)[:500]}",
                    datetime.utcnow().isoformat()
                ))

    print(f"Internal wiki markdown docs found: {len(wiki_rows):,}")
else:
    print(f"Wiki path not found: {WIKI_ROOT}")
    print("Skipping internal wiki ingestion.")

wiki_schema = result_schema

if wiki_rows:
    wiki_df = spark.createDataFrame(wiki_rows, schema=wiki_schema)
    wiki_ok_df = wiki_df.filter(col("status") == "ok")
    wiki_bad_df = wiki_df.filter(col("status") != "ok")

    print(f"✓ Wiki successful : {wiki_ok_df.count():,}")
    print(f"✗ Wiki failed     : {wiki_bad_df.count():,}")

    if wiki_bad_df.count() > 0:
        display(wiki_bad_df)
else:
    wiki_ok_df = spark.createDataFrame([], schema=wiki_schema)


## 11. Write The Combined Raw Corpus

This cell combines the official Microsoft Learn rows with any optional internal wiki rows and overwrites the raw Delta table.

The output table, `chatbot.rag_chatbot.raw_docs`, is the contract consumed by the chunking notebook. If this table is wrong, every downstream chunk, embedding, and Vector Search result inherits the problem.

In [0]:
# ── Combine Microsoft Learn docs + internal wiki docs ───────────────────────
combined_df = ok_df.unionByName(wiki_ok_df)

print(f"Microsoft Learn docs : {ok_df.count():,}")
print(f"Internal wiki docs   : {wiki_ok_df.count():,}")
print(f"Combined docs        : {combined_df.count():,}")

(
    combined_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(RAW_TABLE)
)

print(f"✓ Written to : {RAW_TABLE}")
print(f"✓ Row count  : {spark.table(RAW_TABLE).count():,}")

## 12. Validate The Raw Corpus Shape

This cell prints the final schema, row count, and content-length distribution for the raw docs table.

Use this as a quality gate before running chunking. Very short pages, empty content, or an unexpected schema usually mean the scraper should be fixed before embeddings are generated.

In [0]:
from pyspark.sql.functions import length

df = spark.table(RAW_TABLE)
df.printSchema()

print(f"\nTotal rows: {df.count():,}")

print("\nContent length distribution:")
display(
    df.select(
        "title",
        length(col("content")).alias("content_chars")
    )
    .orderBy(col("content_chars").desc())
)

## 13. Optional Source Distribution Check

This skipped cell is a quick diagnostic for checking how many raw documents came from Microsoft Learn versus the optional internal wiki.

Unskip it when you want to verify source balance without changing any tables.

In [0]:
%skip
display(spark.sql("""
SELECT
  CASE
    WHEN url LIKE 'internal_wiki://%' THEN 'internal_wiki'
    ELSE 'microsoft_learn'
  END AS source_guess,
  COUNT(*) AS docs
FROM chatbot.rag_chatbot.raw_docs
GROUP BY source_guess
"""))

## 14. Optional Downstream Chunk Distribution Check

This skipped cell looks at the downstream `doc_chunks` table rather than the raw table. Use it after running Notebook 2 to confirm the chunk table preserved source metadata for official docs and internal content.

In [0]:
%skip
display(spark.sql("""
SELECT source, source_type, COUNT(*) AS chunks
FROM chatbot.rag_chatbot.doc_chunks
GROUP BY source, source_type
ORDER BY chunks DESC
"""))